In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False, 'axes.spines.right': False,
})
FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

GRADES     = list(range(1, 11))
PROVINCES  = ['Punjab', 'Sindh', 'KPK', 'Balochistan']
GENDERS    = ['Male', 'Female']
N_BOOT     = 5_000

PROV_COLORS = {
    'National':    '#2c2c2c',
    'Punjab':      '#1f77b4',
    'Sindh':       '#d62728',
    'KPK':         '#2ca02c',
    'Balochistan': '#ff7f0e',
}
GENDER_COLORS = {'Male': '#4c72b0', 'Female': '#c44e52'}

## Bootstrap Confidence Intervals — Grade-Level Arithmetic Gap

Three cuts: national (grades 1–10), by province (Punjab, Sindh, KPK, Balochistan), and by gender. Results saved to `outputs/grade_gap_with_ci.csv`.

**Bootstrap design:** government and private students resampled independently with replacement (stratified by school_type); 5,000 resamples; 2.5th/97.5th percentiles; `numpy.random.default_rng` for reproducibility.

In [ ]:
import os

COLS = ['child_id', 'grade', 'school_type', 'province', 'gender', 'arithmetic_score']
raw  = pd.read_csv('../Data:/child_merged.csv', usecols=COLS)

df = raw[
    raw['school_type'].isin(['Government', 'Private']) &
    raw['grade'].between(1, 10) &
    raw['arithmetic_score'].notna()
].copy()
df['grade'] = df['grade'].astype(int)

# Attach VCODES from raw ASER xlsx (absent from child_merged.csv).
# Override path with ASER_CHILD_XLSX environment variable if set.
XLSX_PATH = os.environ.get('ASER_CHILD_XLSX', '../data/raw/ITAASER2023Child.xlsx')
print(f'Loading VCODES from: {XLSX_PATH}')
raw_xl = pd.read_excel(XLSX_PATH, usecols=['Id', 'VCODES'])
raw_xl.columns = ['child_id', 'VCODES']
df = df.merge(raw_xl, on='child_id', how='left')

n_matched = df['VCODES'].notna().sum()
print(f'Rows after filter: {len(df):,}')
print(df.groupby('school_type').size().rename('n').to_frame().T)
print(f"Gender NaN rate: {df['gender'].isna().mean():.1%}")
print(f'VCODES matched: {n_matched:,}/{len(df):,} ({n_matched/len(df):.1%})')

In [ ]:
def bootstrap_gap_ci(govt, pvt, n_boot=N_BOOT, seed=42):
    """
    Percentile bootstrap 95% CI for gap = pvt_mean - govt_mean.
    Resamples government and private groups independently.
    Returns (ci_low, ci_high) or (nan, nan) if either group has < 2 obs.
    """
    g, p = np.asarray(govt, dtype=float), np.asarray(pvt, dtype=float)
    if len(g) < 2 or len(p) < 2:
        return np.nan, np.nan
    rng   = np.random.default_rng(seed)
    idx_g = rng.integers(0, len(g), (n_boot, len(g)))
    idx_p = rng.integers(0, len(p), (n_boot, len(p)))
    gaps  = p[idx_p].mean(axis=1) - g[idx_g].mean(axis=1)
    return float(np.percentile(gaps, 2.5)), float(np.percentile(gaps, 97.5))


def cluster_bootstrap_gap_ci(df_slice, n_boot=N_BOOT, seed=42):
    """
    Cluster bootstrap 95% CI for gap = pvt_mean - govt_mean.
    Resamples whole villages (VCODES) with replacement. Vectorised via
    np.bincount weights — identical approach to notebook 15.
    Returns (ci_low, ci_high) or (nan, nan) if fewer than 2 villages.
    """
    sub = df_slice[df_slice['VCODES'].notna()].copy()
    unique_v = np.unique(sub['VCODES'].values)
    n_v = len(unique_v)
    if n_v < 2:
        return np.nan, np.nan

    vill_map = {v: i for i, v in enumerate(unique_v)}
    vill_int = np.array([vill_map[v] for v in sub['VCODES'].values])

    govt_mask = (sub['school_type'].values == 'Government').astype(float)
    pvt_mask  = (sub['school_type'].values == 'Private').astype(float)
    scores    = sub['arithmetic_score'].values.astype(float)

    rng = np.random.default_rng(seed)
    boot_gaps = []
    for _ in range(n_boot):
        drawn  = rng.integers(0, n_v, n_v)
        counts = np.bincount(drawn, minlength=n_v)
        w      = counts[vill_int]
        wg = w * govt_mask
        wp = w * pvt_mask
        sg, sp = wg.sum(), wp.sum()
        if sg > 0 and sp > 0:
            boot_gaps.append(np.dot(wp, scores) / sp - np.dot(wg, scores) / sg)
    if len(boot_gaps) < 100:
        return np.nan, np.nan
    return float(np.percentile(boot_gaps, 2.5)), float(np.percentile(boot_gaps, 97.5))


def compute_rows(data, province='National', gender='All'):
    """Compute gap + child CI + cluster CI for grades 1–10 in a given province/gender slice."""
    sub = data.copy()
    if province != 'National':
        sub = sub[sub['province'] == province]
    if gender != 'All':
        sub = sub[sub['gender'] == gender]

    rows = []
    for grade in GRADES:
        g = sub[sub['grade'] == grade]
        govt = g[g['school_type'] == 'Government']['arithmetic_score'].values
        pvt  = g[g['school_type'] == 'Private']['arithmetic_score'].values
        if len(govt) < 5 or len(pvt) < 5:
            continue
        gap = float(pvt.mean() - govt.mean())
        ci_low,         ci_high         = bootstrap_gap_ci(govt, pvt)
        cluster_ci_low, cluster_ci_high = cluster_bootstrap_gap_ci(g)
        rows.append({
            'grade':           grade,
            'province':        province,
            'gender':          gender,
            'n_govt':          len(govt),
            'n_pvt':           len(pvt),
            'govt_mean':       round(float(govt.mean()), 4),
            'pvt_mean':        round(float(pvt.mean()),  4),
            'gap':             round(gap, 4),
            'ci_low':          round(ci_low,          4),
            'ci_high':         round(ci_high,         4),
            'cluster_ci_low':  round(cluster_ci_low,  4) if not np.isnan(cluster_ci_low)  else np.nan,
            'cluster_ci_high': round(cluster_ci_high, 4) if not np.isnan(cluster_ci_high) else np.nan,
        })
    return rows


print('Bootstrap functions defined (child-level and cluster).')

In [ ]:
import time
t0 = time.time()
all_rows = []

print('National...')
all_rows.extend(compute_rows(df))

for prov in PROVINCES:
    print(f'{prov}...')
    all_rows.extend(compute_rows(df, province=prov))

for gen in GENDERS:
    print(f'{gen} (national)...')
    all_rows.extend(compute_rows(df, gender=gen))

results = pd.DataFrame(all_rows)
results.to_csv('../outputs/grade_gap_with_ci.csv', index=False)
print(f'\nDone in {time.time()-t0:.1f}s  |  Saved {len(results)} rows → outputs/grade_gap_with_ci.csv\n')

# National summary table with both CIs
nat = results[(results['province'] == 'National') & (results['gender'] == 'All')]
print('National gaps with child and cluster 95% CIs:')
print(nat[['grade','n_govt','n_pvt','gap',
           'ci_low','ci_high','cluster_ci_low','cluster_ci_high']].to_string(index=False))

In [ ]:
# ── Before/after: child CI vs cluster CI for national line ──
nat = results[(results['province'] == 'National') & (results['gender'] == 'All')].sort_values('grade')

print('National grade-gap CIs — child-level vs cluster bootstrap')
print('-' * 95)
print(f"{'grade':>5}  {'n_govt':>6}  {'n_pvt':>6}  {'gap':>7}  "
      f"{'child_lo':>9}  {'child_hi':>9}  {'clust_lo':>9}  {'clust_hi':>9}  {'ratio':>6}")
print('-' * 95)
for _, row in nat.iterrows():
    cw    = row['ci_high']          - row['ci_low']
    kw    = row['cluster_ci_high']  - row['cluster_ci_low']
    ratio = kw / cw if cw > 0 else float('nan')
    print(f"{int(row['grade']):>5}  {int(row['n_govt']):>6}  {int(row['n_pvt']):>6}  "
          f"{row['gap']:>7.3f}  "
          f"{row['ci_low']:>9.3f}  {row['ci_high']:>9.3f}  "
          f"{row['cluster_ci_low']:>9.3f}  {row['cluster_ci_high']:>9.3f}  "
          f"{ratio:>6.2f}x")
print('-' * 95)

g5 = nat[nat['grade'] == 5].iloc[0]
child_g5_excl   = not (g5['ci_low']          <= 0 <= g5['ci_high'])
cluster_g5_excl = not (g5['cluster_ci_low']  <= 0 <= g5['cluster_ci_high'])
print(f"\nGrade-5 gap = {g5['gap']:.3f}")
print(f"  Child CI   [{g5['ci_low']:.3f}, {g5['ci_high']:.3f}]  → excludes zero: {child_g5_excl}")
print(f"  Cluster CI [{g5['cluster_ci_low']:.3f}, {g5['cluster_ci_high']:.3f}]  → excludes zero: {cluster_g5_excl}")

## Figure — Government–Private Arithmetic Gap by Grade (with 95% Cluster Bootstrap CI Bands)

Shaded bands: 95% cluster bootstrap CI. Cells with n_pvt < 100 are suppressed (line breaks).

In [ ]:
MIN_PVT = 100  # presentation threshold: hide (province, grade) cells with n_pvt < 100
               # (statistical analysis in this notebook retains the n >= 50 threshold)

# Identify and report excluded cells before drawing
excluded = []
for group in ['National'] + PROVINCES:
    sub_all = results[(results['province'] == group) & (results['gender'] == 'All')]
    for _, row in sub_all.iterrows():
        if row['n_pvt'] < MIN_PVT:
            excluded.append((group, int(row['grade']), int(row['n_pvt'])))

print('Excluded (province, grade) cells (n_pvt < 100):')
if excluded:
    for prov, gr, n in excluded:
        print(f'  {prov} grade {gr}: n_pvt = {n}')
else:
    print('  None')
print()

fig, ax = plt.subplots(figsize=(10, 6))

for group in ['National'] + PROVINCES:
    color = PROV_COLORS[group]
    sub = (
        results[(results['province'] == group) & (results['gender'] == 'All')]
        .sort_values('grade')
        .set_index('grade')
        .reindex(GRADES)   # ensure all grades 1–10 are present (NaN for missing)
    )

    # Mask cells below the presentation threshold — NaN breaks the line automatically
    thin_mask = sub['n_pvt'] < MIN_PVT
    sub.loc[thin_mask, ['gap', 'cluster_ci_low', 'cluster_ci_high']] = np.nan

    grades = sub.index.values
    gap    = sub['gap'].values
    ci_low = sub['cluster_ci_low'].values
    ci_hi  = sub['cluster_ci_high'].values

    lw = 2.5 if group == 'National' else 1.8
    ls = '--' if group == 'National' else '-'

    ax.plot(grades, gap,
            color=color, linewidth=lw, linestyle=ls, label=group, zorder=4)

    # fill_between breaks naturally at NaN in both y1 and y2
    ax.fill_between(grades, ci_low, ci_hi,
                    color=color, alpha=0.12, zorder=3)

ax.axhline(0, color='#888888', linewidth=0.9, linestyle=':', zorder=2)
ax.set_xlabel('Grade', labelpad=8)
ax.set_ylabel('Arithmetic gap (private − government)', labelpad=8)
ax.set_title(
    'Government–Private Arithmetic Gap by Grade and Province\n'
    'with 95% Cluster Bootstrap Confidence Intervals',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xticks(GRADES)
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.grid(axis='x', visible=False)
ax.set_axisbelow(True)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '01b_gap_by_grade_with_ci.png', bbox_inches='tight')
fig.savefig(FIGURES / '01b_gap_by_grade_with_ci.pdf', bbox_inches='tight')
plt.show()
print('Saved → outputs/figures/01b_gap_by_grade_with_ci.png  (cluster bootstrap CIs)')

## Gender Hypothesis Test

$\Delta = \bar{\text{gap}}_{\text{girls}} - \bar{\text{gap}}_{\text{boys}}$, averaged across grades 1–10.

**Bootstrap design:** government and private groups resampled independently within each grade × gender cell; 5,000 resamples; 2.5th/97.5th percentiles on $\Delta$.

In [ ]:
# ── Observed statistic ──
girls_gaps = results[(results['gender'] == 'Female') & (results['province'] == 'National')]
boys_gaps  = results[(results['gender'] == 'Male')   & (results['province'] == 'National')]

obs_girls = float(girls_gaps['gap'].mean())
obs_boys  = float(boys_gaps['gap'].mean())
obs_delta = obs_girls - obs_boys

print(f'Observed mean gap (grades 1–10):')
print(f'  Girls: {obs_girls:.4f}')
print(f'  Boys:  {obs_boys:.4f}')
print(f'  Delta (girls − boys): {obs_delta:+.4f}')

# ═══════════════════════════════════════════════════════
# Child-level bootstrap (original design)
# ═══════════════════════════════════════════════════════
score_cache = {}
for grade in GRADES:
    g = df[df['grade'] == grade]
    for gender in GENDERS:
        for stype in ['Government', 'Private']:
            score_cache[(grade, gender, stype)] = g[
                (g['gender'] == gender) & (g['school_type'] == stype)
            ]['arithmetic_score'].values

rng = np.random.default_rng(99)
boot_deltas_child = np.empty(N_BOOT)

for i in range(N_BOOT):
    female_gaps, male_gaps = [], []
    for grade in GRADES:
        fg = score_cache[(grade, 'Female', 'Government')]
        fp = score_cache[(grade, 'Female', 'Private')]
        mg = score_cache[(grade, 'Male',   'Government')]
        mp = score_cache[(grade, 'Male',   'Private')]
        if len(fg) > 1 and len(fp) > 1:
            female_gaps.append(fp[rng.integers(0, len(fp), len(fp))].mean() -
                               fg[rng.integers(0, len(fg), len(fg))].mean())
        if len(mg) > 1 and len(mp) > 1:
            male_gaps.append(mp[rng.integers(0, len(mp), len(mp))].mean() -
                             mg[rng.integers(0, len(mg), len(mg))].mean())
    boot_deltas_child[i] = np.mean(female_gaps) - np.mean(male_gaps)

child_ci_low  = float(np.percentile(boot_deltas_child, 2.5))
child_ci_high = float(np.percentile(boot_deltas_child, 97.5))

# ═══════════════════════════════════════════════════════
# Cluster bootstrap (village-level resampling)
# ═══════════════════════════════════════════════════════
# Build village × cell aggregation matrices for efficient bootstrap.
# Cells: grade (10) × gender (2) × school_type (2) = 40 cells.
# Index: grade_idx*4 + gender_idx*2 + st_idx  (0=Govt, 1=Pvt)
clust_sub = df[
    df['VCODES'].notna() &
    df['school_type'].isin(['Government', 'Private']) &
    df['arithmetic_score'].notna() &
    df['gender'].isin(['Male', 'Female'])
].copy()

unique_v = np.unique(clust_sub['VCODES'].values)
n_v      = len(unique_v)
vill_map = {v: i for i, v in enumerate(unique_v)}
vill_idx = np.array([vill_map[v] for v in clust_sub['VCODES'].values])

grades_arr = clust_sub['grade'].values.astype(int)
school_arr = clust_sub['school_type'].values
gender_arr = clust_sub['gender'].values
scores_arr = clust_sub['arithmetic_score'].values.astype(float)

grade_vals = list(range(1, 11))
gen_vals   = ['Female', 'Male']
n_cells    = 40  # 10 grades × 2 genders × 2 school types

cell_idx = np.full(len(clust_sub), -1, dtype=int)
for gi, g in enumerate(grade_vals):
    for geni, gen in enumerate(gen_vals):
        for sti, st in enumerate(['Government', 'Private']):
            ci = gi * 4 + geni * 2 + sti
            cell_idx[(grades_arr == g) & (gender_arr == gen) & (school_arr == st)] = ci

valid    = cell_idx >= 0
lin_idx  = vill_idx[valid] * n_cells + cell_idx[valid]
score_sums = np.bincount(lin_idx, weights=scores_arr[valid], minlength=n_v * n_cells).reshape(n_v, n_cells)
count_sums = np.bincount(lin_idx,                            minlength=n_v * n_cells).reshape(n_v, n_cells).astype(float)

rng_c = np.random.default_rng(99)
boot_deltas_cluster = []

for _ in range(N_BOOT):
    drawn  = rng_c.integers(0, n_v, n_v)
    w      = np.bincount(drawn, minlength=n_v)
    tot_sc = w @ score_sums   # (n_cells,)
    tot_ct = w @ count_sums   # (n_cells,)

    female_gaps, male_gaps = [], []
    for gi in range(len(grade_vals)):
        for geni, gap_list in enumerate([female_gaps, male_gaps]):
            govt_ci = gi * 4 + geni * 2 + 0
            pvt_ci  = gi * 4 + geni * 2 + 1
            sg = tot_ct[govt_ci]
            sp = tot_ct[pvt_ci]
            if sg > 0 and sp > 0:
                gap_list.append(tot_sc[pvt_ci] / sp - tot_sc[govt_ci] / sg)

    if female_gaps and male_gaps:
        boot_deltas_cluster.append(np.mean(female_gaps) - np.mean(male_gaps))

cluster_ci_low  = float(np.percentile(boot_deltas_cluster, 2.5))
cluster_ci_high = float(np.percentile(boot_deltas_cluster, 97.5))

# ═══════════════════════════════════════════════════════
# Before / after comparison
# ═══════════════════════════════════════════════════════
print(f'\n{"─"*60}')
print(f'Gender delta CI — before/after cluster bootstrap')
print(f'{"─"*60}')
print(f'Delta (girls − boys):   {obs_delta:+.4f}')
print(f'Child CI:    [{child_ci_low:+.4f}, {child_ci_high:+.4f}]  '
      f'crosses zero: {child_ci_low < 0 < child_ci_high}')
print(f'Cluster CI:  [{cluster_ci_low:+.4f}, {cluster_ci_high:+.4f}]  '
      f'crosses zero: {cluster_ci_low < 0 < cluster_ci_high}')

In [ ]:
# Per-grade breakdown of girls' vs boys' gaps
gender_tbl = (
    results[results['province'] == 'National']
    .query("gender in ['Male','Female']")
    .pivot(index='grade', columns='gender', values=['gap','ci_low','ci_high'])
)
gender_tbl.columns = [f'{c[0]}_{c[1].lower()}' for c in gender_tbl.columns]
gender_tbl['delta'] = (gender_tbl['gap_female'] - gender_tbl['gap_male']).round(4)
print('Per-grade girls vs boys gap:')
print(gender_tbl[['gap_female','ci_low_female','ci_high_female',
                   'gap_male',  'ci_low_male',  'ci_high_male', 'delta']].round(3).to_string())